# Model-in-the-Loop Validation — Supplemental Experiments

This notebook runs supplemental model-in-the-loop experiments for the self-healing orchestration study.

The notebook complements the main 15-task validation with two focused analyses:

- **Verifier ablation**: compares self-healing with tool-output verification enabled versus disabled.
- **Recovery-budget sensitivity**: measures how success, recovery, and call-count overhead change as the recovery budget varies.

The tools remain local and deterministic. A live tool-calling language model performs tool selection, tool argument generation, and final answer synthesis while controlled local tool-output faults are injected.


## How to run

Before running the notebook, provide an API key through an environment variable or a private notebook secret. Do not commit API keys to GitHub.

For Google Colab, prefer Colab Secrets:

```python
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
```

Then set the live configuration:

```python
os.environ["FORCE_MOCK_MODEL"] = "false"
os.environ["OPENAI_MODEL"] = "gpt-5.4-mini"
os.environ["OPENAI_VERIFIER_MODEL"] = os.environ["OPENAI_MODEL"]
os.environ["USE_LLM_EVALUATOR"] = "true"
```

Optional cost-control switches:

```python
# Run only the verifier ablation.
os.environ["RUN_BUDGET_SENSITIVITY"] = "false"

# Reduce the budget-sensitivity task subset.
os.environ["BUDGET_TASK_LIMIT"] = "5"
```

Default run sizes:

```text
Verifier ablation: 15 tasks × 2 variants × 2 seeds = 60 runs
Budget sensitivity: 10 tasks × 3 methods × 3 budgets × 2 seeds = 180 runs
```


In [ ]:
# Optional Colab dependency install. Uncomment if needed.
%pip install -q openai python-dotenv pandas numpy matplotlib


In [ ]:
from __future__ import annotations

import os
import re
import json
import time
import math
import random
import hashlib
from copy import deepcopy
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 240)
pd.set_option("display.max_colwidth", 220)

In [ ]:
# ============================================================
# Configuration
# ============================================================

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
OPENAI_VERIFIER_MODEL = os.getenv("OPENAI_VERIFIER_MODEL", OPENAI_MODEL)

FORCE_MOCK_MODEL = os.getenv("FORCE_MOCK_MODEL", "true").lower() in {"1", "true", "yes"}
USE_LLM_EVALUATOR = os.getenv("USE_LLM_EVALUATOR", "true").lower() in {"1", "true", "yes"}
HAS_OPENAI_KEY = bool(os.getenv("OPENAI_API_KEY"))
USE_MOCK_MODEL = FORCE_MOCK_MODEL or not HAS_OPENAI_KEY

MAX_AGENT_STEPS = int(os.getenv("MAX_AGENT_STEPS", "8"))
RECOVERY_BUDGET = int(os.getenv("RECOVERY_BUDGET", "2"))

SUPPLEMENTAL_SEEDS = [
    int(x.strip())
    for x in os.getenv("SUPPLEMENTAL_SEEDS", "11,22").split(",")
    if x.strip()
]

LIVE_TASK_LIMIT_ENV = os.getenv("LIVE_TASK_LIMIT", "").strip()
LIVE_TASK_LIMIT = int(LIVE_TASK_LIMIT_ENV) if LIVE_TASK_LIMIT_ENV else None

# Fault set used in the model-in-the-loop validation.
FAULT_INTENSITY = float(os.getenv("FAULT_INTENSITY", "0.3"))
METHODS = ["retry_only", "full_replanning", "self_healing"]
FAULT_TYPES = [
    "timeout",
    "unavailable_tool",
    "malformed_output",
    "stale_context",
    "wrong_but_plausible",
]

# Supplemental experiment switches.
RUN_VERIFIER_ABLATION = os.getenv("RUN_VERIFIER_ABLATION", "true").lower() in {"1", "true", "yes"}
RUN_BUDGET_SENSITIVITY = os.getenv("RUN_BUDGET_SENSITIVITY", "true").lower() in {"1", "true", "yes"}

BUDGET_LEVELS = [
    int(x.strip())
    for x in os.getenv("BUDGET_LEVELS", "1,2,3").split(",")
    if x.strip()
]
BUDGET_TASK_LIMIT_ENV = os.getenv("BUDGET_TASK_LIMIT", "10").strip()
BUDGET_TASK_LIMIT = int(BUDGET_TASK_LIMIT_ENV) if BUDGET_TASK_LIMIT_ENV else 10

# Tool-output verification is toggled by the supplemental experiment runner.
TOOL_OUTPUT_VERIFIER_ENABLED = True

# Optional pricing. Keep raw token counts regardless of USD estimate.
INPUT_COST_PER_MILLION = float(os.getenv("INPUT_COST_PER_MILLION", "0"))
OUTPUT_COST_PER_MILLION = float(os.getenv("OUTPUT_COST_PER_MILLION", "0"))

BASE_DIR = Path.cwd()
RESULTS_DIR = BASE_DIR / "results"
FIGURES_DIR = BASE_DIR / "figures"
TABLES_DIR = BASE_DIR / "tables"
TRACES_DIR = BASE_DIR / "traces"

for d in [RESULTS_DIR, FIGURES_DIR, TABLES_DIR, TRACES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Configuration")
print("-------------")
print("OPENAI_MODEL:", OPENAI_MODEL)
print("OPENAI_VERIFIER_MODEL:", OPENAI_VERIFIER_MODEL)
print("USE_MOCK_MODEL:", USE_MOCK_MODEL)
print("USE_LLM_EVALUATOR:", USE_LLM_EVALUATOR)
print("FAULT_INTENSITY:", FAULT_INTENSITY)
print("FAULT_TYPES:", FAULT_TYPES)
print("SUPPLEMENTAL_SEEDS:", SUPPLEMENTAL_SEEDS)
print("LIVE_TASK_LIMIT:", LIVE_TASK_LIMIT)
print("RUN_VERIFIER_ABLATION:", RUN_VERIFIER_ABLATION)
print("RUN_BUDGET_SENSITIVITY:", RUN_BUDGET_SENSITIVITY)
print("BUDGET_LEVELS:", BUDGET_LEVELS)
print("BUDGET_TASK_LIMIT:", BUDGET_TASK_LIMIT)
print("Output dirs:", RESULTS_DIR, FIGURES_DIR, TABLES_DIR, TRACES_DIR)


## Local tool environment

The model is live, but the tools are local and deterministic. A controlled **fault injector** wraps tool outputs. This lets us evaluate orchestration behavior under matched fault conditions while keeping the external tool environment reproducible.


In [ ]:
# ============================================================
# Local data used by deterministic tools
# ============================================================

ORDERS = {
    "O-1001": {"order_id": "O-1001", "customer_id": "C-001", "sku": "SKU-A", "status": "delayed", "delay_hours": 60, "shipped": True, "damaged": False, "days_since_purchase": 14},
    "O-1002": {"order_id": "O-1002", "customer_id": "C-002", "sku": "SKU-B", "status": "delayed", "delay_hours": 72, "shipped": True, "damaged": False, "days_since_purchase": 20},
    "O-1003": {"order_id": "O-1003", "customer_id": "C-003", "sku": "SKU-C", "status": "on_time", "delay_hours": 0, "shipped": True, "damaged": False, "days_since_purchase": 5},
    "O-1004": {"order_id": "O-1004", "customer_id": "C-004", "sku": "SKU-D", "status": "delayed", "delay_hours": 52, "shipped": True, "damaged": True, "days_since_purchase": 9},
    "O-1005": {"order_id": "O-1005", "customer_id": "C-005", "sku": "SKU-E", "status": "delayed", "delay_hours": 80, "shipped": True, "damaged": False, "days_since_purchase": 40},
    "O-2001": {"order_id": "O-2001", "customer_id": "C-001", "sku": "SKU-A", "status": "processing", "delay_hours": 0, "destination": "standard_zone", "requested_shipping": "priority", "quantity": 2},
    "O-2002": {"order_id": "O-2002", "customer_id": "C-002", "sku": "SKU-B", "status": "processing", "delay_hours": 0, "destination": "restricted_zone", "requested_shipping": "priority", "quantity": 1},
    "O-2003": {"order_id": "O-2003", "customer_id": "C-003", "sku": "SKU-C", "status": "processing", "delay_hours": 0, "destination": "standard_zone", "requested_shipping": "standard", "quantity": 3},
    "O-2004": {"order_id": "O-2004", "customer_id": "C-004", "sku": "SKU-D", "status": "processing", "delay_hours": 0, "destination": "standard_zone", "requested_shipping": "priority", "quantity": 5},
    "O-2005": {"order_id": "O-2005", "customer_id": "C-005", "sku": "SKU-E", "status": "processing", "delay_hours": 0, "destination": "standard_zone", "requested_shipping": "priority", "quantity": 1},
}

CUSTOMERS = {
    "C-001": {"customer_id": "C-001", "tier": "Gold", "status": "active"},
    "C-002": {"customer_id": "C-002", "tier": "Silver", "status": "active"},
    "C-003": {"customer_id": "C-003", "tier": "Platinum", "status": "active"},
    "C-004": {"customer_id": "C-004", "tier": "Gold", "status": "active"},
    "C-005": {"customer_id": "C-005", "tier": "Gold", "status": "inactive"},
}

INVENTORY = {
    "SKU-A": {"sku": "SKU-A", "in_stock": True, "available_units": 25, "warehouse": "W1"},
    "SKU-B": {"sku": "SKU-B", "in_stock": True, "available_units": 10, "warehouse": "W2"},
    "SKU-C": {"sku": "SKU-C", "in_stock": True, "available_units": 4, "warehouse": "W1"},
    "SKU-D": {"sku": "SKU-D", "in_stock": False, "available_units": 0, "warehouse": "W3"},
    "SKU-E": {"sku": "SKU-E", "in_stock": True, "available_units": 8, "warehouse": "W2"},
}

POLICIES = {
    "replacement_policy": {
        "policy_name": "replacement_policy",
        "version": "2026-05-current",
        "text": "Expedited replacement is allowed when an order is delayed by more than 48 hours, the customer is Gold or Platinum tier, and the customer account is active at evaluation time. Inactive accounts are not eligible for expedited replacement.",
        "effective_date": "2026-05-01",
    },
    "shipping_policy": {
        "policy_name": "shipping_policy",
        "version": "2026-05-current",
        "text": "Priority shipping is available only when the requested item is in stock, requested quantity is available, and destination is not a restricted zone. Otherwise standard shipping is required.",
        "effective_date": "2026-05-01",
    },
    "return_policy": {
        "policy_name": "return_policy",
        "version": "2026-05-current",
        "text": "Returns are allowed within 30 days of purchase. Damaged items are eligible for return review even if shipped.",
        "effective_date": "2026-05-01",
    },
}


In [ ]:
# ============================================================
# Deterministic local tools
# ============================================================

def ok(data: dict, source: str) -> dict:
    """Return a standardized successful tool response.

    Args:
        data: Tool-specific payload to return.
        source: Name of the tool that produced the response.

    Returns:
        A dictionary with a consistent tool-response schema. The response is
        marked as current, unfaulted evidence and includes a deep copy of the
        payload to avoid accidental mutation of the source fixture data.
    """
    return {
        "status": "ok",
        "data": deepcopy(data),
        "evidence_source": source,
        "freshness": "current",
        "fault_injected": None,
    }


def error(message: str, source: str) -> dict:
    """Return a standardized not-found tool error.

    Args:
        message: Human-readable error message.
        source: Name of the tool that produced the error.

    Returns:
        A dictionary with a consistent error-response schema. These errors
        represent ordinary lookup failures, not injected experimental faults.
    """
    return {
        "status": "error",
        "error_type": "not_found",
        "message": message,
        "evidence_source": source,
        "fault_injected": None,
    }


def order_lookup(order_id: str) -> dict:
    """Look up an order record by order identifier.

    Args:
        order_id: Identifier of the order to retrieve.

    Returns:
        A standardized successful response containing the order record when the
        order exists, otherwise a standardized not-found error.
    """
    if order_id in ORDERS:
        return ok(ORDERS[order_id], "order_lookup")
    return error(f"Unknown order_id {order_id}", "order_lookup")


def customer_lookup(customer_id: str) -> dict:
    """Look up a customer record by customer identifier.

    Args:
        customer_id: Identifier of the customer to retrieve.

    Returns:
        A standardized successful response containing the customer record when
        the customer exists, otherwise a standardized not-found error.
    """
    if customer_id in CUSTOMERS:
        return ok(CUSTOMERS[customer_id], "customer_lookup")
    return error(f"Unknown customer_id {customer_id}", "customer_lookup")


def inventory_lookup(sku: str) -> dict:
    """Look up inventory state for a product SKU.

    Args:
        sku: Product stock-keeping unit to retrieve.

    Returns:
        A standardized successful response containing inventory information
        when the SKU exists, otherwise a standardized not-found error.
    """
    if sku in INVENTORY:
        return ok(INVENTORY[sku], "inventory_lookup")
    return error(f"Unknown sku {sku}", "inventory_lookup")


def policy_lookup(policy_name: str) -> dict:
    """Look up the current policy text for a named policy.

    Args:
        policy_name: Name of the policy to retrieve.

    Returns:
        A standardized successful response containing current policy evidence
        when the policy exists, otherwise a standardized not-found error.
    """
    if policy_name in POLICIES:
        return ok(POLICIES[policy_name], "policy_lookup")
    return error(f"Unknown policy_name {policy_name}", "policy_lookup")


def shipping_estimator(
    sku: str,
    destination: str,
    requested_shipping: str = "standard",
    quantity: int = 1,
) -> dict:
    """Estimate whether a shipping request can be fulfilled.

    The estimator uses deterministic inventory and destination rules. It is
    intentionally simple so that downstream failures can be attributed to the
    orchestration and fault-injection logic rather than external API behavior.

    Args:
        sku: Product stock-keeping unit.
        destination: Requested destination region.
        requested_shipping: Requested shipping class, such as "standard" or
            "priority".
        quantity: Number of units requested.

    Returns:
        A standardized response containing whether the requested shipping path
        is allowed, the reason for the decision, and an estimated delivery time.
        Unknown SKUs return a standardized not-found error.
    """
    inv = INVENTORY.get(sku)
    if not inv:
        return error(f"Unknown sku {sku}", "shipping_estimator")

    if destination == "restricted_zone":
        allowed = False
        reason = "destination is restricted"
    elif not inv["in_stock"] or inv["available_units"] < quantity:
        allowed = False
        reason = "inventory is insufficient"
    elif requested_shipping == "priority":
        allowed = True
        reason = "priority shipping available"
    else:
        allowed = True
        reason = "standard shipping available"

    estimated_days = 2 if allowed and requested_shipping == "priority" else 5

    return ok(
        {
            "sku": sku,
            "destination": destination,
            "requested_shipping": requested_shipping,
            "quantity": quantity,
            "priority_allowed": allowed,
            "reason": reason,
            "estimated_days": estimated_days,
        },
        "shipping_estimator",
    )


def calculator(expression: str) -> dict:
    """Evaluate a restricted arithmetic expression.

    This calculator is used only for controlled benchmark expressions. It
    accepts numeric arithmetic containing digits, decimal points, parentheses,
    whitespace, and the operators +, -, *, and /. Non-arithmetic expressions are
    rejected before evaluation.

    Args:
        expression: Arithmetic expression to evaluate.

    Returns:
        A standardized successful response containing the numeric value, or a
        standardized calculator error if the expression is unsafe or invalid.
    """
    if not re.fullmatch(r"[0-9\.\+\-\*\/\(\)\s]+", expression):
        return {
            "status": "error",
            "error_type": "unsafe_expression",
            "message": "Only numeric arithmetic is allowed",
            "evidence_source": "calculator",
            "fault_injected": None,
        }

    try:
        value = eval(expression, {"__builtins__": {}}, {})
        return ok({"expression": expression, "value": float(value)}, "calculator")
    except Exception as exc:
        return {
            "status": "error",
            "error_type": "calculation_error",
            "message": str(exc),
            "evidence_source": "calculator",
            "fault_injected": None,
        }


TOOL_IMPLS = {
    "order_lookup": order_lookup,
    "customer_lookup": customer_lookup,
    "inventory_lookup": inventory_lookup,
    "policy_lookup": policy_lookup,
    "shipping_estimator": shipping_estimator,
    "calculator": calculator,
}

In [ ]:
# ============================================================
# OpenAI function/tool schemas
# ============================================================

OPENAI_TOOLS = [
    {"type": "function", "function": {"name": "order_lookup", "description": "Look up order status, customer id, SKU, delay, destination, quantity, and shipping request details.", "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"], "additionalProperties": False}}},
    {"type": "function", "function": {"name": "customer_lookup", "description": "Look up customer tier and account status.", "parameters": {"type": "object", "properties": {"customer_id": {"type": "string"}}, "required": ["customer_id"], "additionalProperties": False}}},
    {"type": "function", "function": {"name": "inventory_lookup", "description": "Look up inventory availability for a SKU.", "parameters": {"type": "object", "properties": {"sku": {"type": "string"}}, "required": ["sku"], "additionalProperties": False}}},
    {"type": "function", "function": {"name": "policy_lookup", "description": "Look up current policy text by policy name. Valid names include replacement_policy, shipping_policy, return_policy.", "parameters": {"type": "object", "properties": {"policy_name": {"type": "string", "enum": ["replacement_policy", "shipping_policy", "return_policy"]}}, "required": ["policy_name"], "additionalProperties": False}}},
    {"type": "function", "function": {"name": "shipping_estimator", "description": "Estimate whether requested shipping is allowed given SKU, destination, requested shipping, and quantity.", "parameters": {"type": "object", "properties": {"sku": {"type": "string"}, "destination": {"type": "string"}, "requested_shipping": {"type": "string", "enum": ["standard", "priority"]}, "quantity": {"type": "integer"}}, "required": ["sku", "destination", "requested_shipping", "quantity"], "additionalProperties": False}}},
    {"type": "function", "function": {"name": "calculator", "description": "Evaluate a simple arithmetic expression.", "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"], "additionalProperties": False}}},
]


## Calibrated 15-task validation set

This notebook uses the same calibrated 15-task subset as the main model-in-the-loop validation. The subset spans retrieval/evidence, multi-step API workflow, and calculation/verification tasks. Controlled tool-output faults are injected to evaluate recovery behavior under matched experimental conditions.


In [ ]:
# ============================================================
# Calibration task set: 15 tasks / 3 categories
# ============================================================

LIVE_TASKS: List[Dict[str, Any]] = [
    # Retrieval / evidence decisions
    {"task_id": "live_retrieval_001", "category": "retrieval_evidence", "success_mode": "decision", "user_request": "Determine whether expedited replacement is allowed for order O-1001. You must use order_lookup, customer_lookup, and policy_lookup evidence.", "expected_decision": "allowed", "expected_facts": ["O-1001 delay_hours is 60", "C-001 tier is Gold", "C-001 status is active", "replacement policy allows eligible active Gold or Platinum customers after >48h delay"], "required_tools": ["order_lookup", "customer_lookup", "policy_lookup"]},
    {"task_id": "live_retrieval_002", "category": "retrieval_evidence", "success_mode": "decision", "user_request": "Determine whether expedited replacement is allowed for order O-1002. You must use order_lookup, customer_lookup, and policy_lookup evidence.", "expected_decision": "not_allowed", "expected_facts": ["O-1002 delay_hours is 72", "C-002 tier is Silver", "replacement policy requires Gold or Platinum"], "required_tools": ["order_lookup", "customer_lookup", "policy_lookup"]},
    {"task_id": "live_retrieval_003", "category": "retrieval_evidence", "success_mode": "decision", "user_request": "Determine whether expedited replacement is allowed for order O-1003. You must use order_lookup, customer_lookup, and policy_lookup evidence.", "expected_decision": "not_allowed", "expected_facts": ["O-1003 delay_hours is 0", "replacement policy requires delay > 48h"], "required_tools": ["order_lookup", "customer_lookup", "policy_lookup"]},
    {"task_id": "live_retrieval_004", "category": "retrieval_evidence", "success_mode": "decision", "user_request": "Determine whether expedited replacement is allowed for order O-1004. You must use order_lookup, customer_lookup, and policy_lookup evidence.", "expected_decision": "allowed", "expected_facts": ["O-1004 delay_hours is 52", "C-004 tier is Gold", "C-004 status is active"], "required_tools": ["order_lookup", "customer_lookup", "policy_lookup"]},
    {"task_id": "live_retrieval_005", "category": "retrieval_evidence", "success_mode": "decision", "user_request": "Determine whether expedited replacement is allowed for order O-1005. You must use order_lookup, customer_lookup, and policy_lookup evidence. Remember that inactive customer accounts are not eligible under the current replacement policy.", "expected_decision": "not_allowed", "expected_facts": ["O-1005 delay_hours is 80", "C-005 status is inactive", "replacement policy requires active account"], "required_tools": ["order_lookup", "customer_lookup", "policy_lookup"]},

    # Multi-step workflow decisions
    {"task_id": "live_workflow_001", "category": "multi_step_api_workflow", "success_mode": "decision", "user_request": "Determine whether priority shipping is allowed for order O-2001. You must use order_lookup, inventory_lookup, shipping_estimator, and policy_lookup evidence.", "expected_decision": "allowed", "expected_facts": ["SKU-A is in stock", "destination is standard_zone", "priority shipping is available"], "required_tools": ["order_lookup", "inventory_lookup", "shipping_estimator", "policy_lookup"]},
    {"task_id": "live_workflow_002", "category": "multi_step_api_workflow", "success_mode": "decision", "user_request": "Determine whether priority shipping is allowed for order O-2002. You must use order_lookup, inventory_lookup, shipping_estimator, and policy_lookup evidence.", "expected_decision": "not_allowed", "expected_facts": ["destination is restricted_zone", "shipping policy blocks priority shipping to restricted zones"], "required_tools": ["order_lookup", "inventory_lookup", "shipping_estimator", "policy_lookup"]},
    {"task_id": "live_workflow_003", "category": "multi_step_api_workflow", "success_mode": "decision", "user_request": "Determine whether the requested standard shipping path is available for order O-2003. You must use order_lookup, inventory_lookup, shipping_estimator, and policy_lookup evidence.", "expected_decision": "allowed", "expected_facts": ["SKU-C has 4 units", "quantity is 3", "standard shipping is available"], "required_tools": ["order_lookup", "inventory_lookup", "shipping_estimator", "policy_lookup"]},
    {"task_id": "live_workflow_004", "category": "multi_step_api_workflow", "success_mode": "decision", "user_request": "Determine whether priority shipping is allowed for order O-2004. You must use order_lookup, inventory_lookup, shipping_estimator, and policy_lookup evidence.", "expected_decision": "not_allowed", "expected_facts": ["SKU-D is out of stock", "priority shipping requires in-stock inventory"], "required_tools": ["order_lookup", "inventory_lookup", "shipping_estimator", "policy_lookup"]},
    {"task_id": "live_workflow_005", "category": "multi_step_api_workflow", "success_mode": "decision", "user_request": "Determine whether priority shipping is allowed for order O-2005. You must use order_lookup, inventory_lookup, shipping_estimator, and policy_lookup evidence.", "expected_decision": "allowed", "expected_facts": ["SKU-E is in stock", "quantity 1 is available", "destination is standard_zone"], "required_tools": ["order_lookup", "inventory_lookup", "shipping_estimator", "policy_lookup"]},

    # Calculation / verification tasks
    {"task_id": "live_calc_001", "category": "calculation_verification", "success_mode": "numeric", "user_request": "Calculate the total cost for 3 items at 19.99 each plus 8.25 percent tax. Use calculator and return numeric_answer.", "expected_decision": "numeric_value", "expected_numeric_answer": round(3 * 19.99 * 1.0825, 2), "numeric_tolerance": 0.02, "required_tools": ["calculator"]},
    {"task_id": "live_calc_002", "category": "calculation_verification", "success_mode": "numeric", "user_request": "Calculate the refund amount for 2 items at 45.50 each with a 15 percent restocking fee deducted. Use calculator and return numeric_answer.", "expected_decision": "numeric_value", "expected_numeric_answer": round(2 * 45.50 * 0.85, 2), "numeric_tolerance": 0.02, "required_tools": ["calculator"]},
    {"task_id": "live_calc_003", "category": "calculation_verification", "success_mode": "numeric", "user_request": "Calculate the weighted score: 0.4 times 82 plus 0.6 times 91. Use calculator and return numeric_answer.", "expected_decision": "numeric_value", "expected_numeric_answer": round(0.4 * 82 + 0.6 * 91, 2), "numeric_tolerance": 0.02, "required_tools": ["calculator"]},
    {"task_id": "live_calc_004", "category": "calculation_verification", "success_mode": "numeric", "user_request": "Calculate the inventory coverage ratio: available units 25 divided by requested units 5. Use calculator and return numeric_answer.", "expected_decision": "numeric_value", "expected_numeric_answer": 5.0, "numeric_tolerance": 0.001, "required_tools": ["calculator"]},
    {"task_id": "live_calc_005", "category": "calculation_verification", "success_mode": "numeric", "user_request": "Calculate final price after a 20 percent discount on 125.00 and then 8 percent tax. Use calculator and return numeric_answer.", "expected_decision": "numeric_value", "expected_numeric_answer": round(125.00 * 0.80 * 1.08, 2), "numeric_tolerance": 0.02, "required_tools": ["calculator"]},
]

if LIVE_TASK_LIMIT is not None:
    LIVE_TASKS = LIVE_TASKS[:LIVE_TASK_LIMIT]

print(f"Loaded {len(LIVE_TASKS)} calibration tasks")
pd.DataFrame([{k: v for k, v in t.items() if k not in {"expected_facts"}} for t in LIVE_TASKS])


In [ ]:
# ============================================================
# Decision normalization and parsing helpers
# ============================================================

DECISION_ALIASES = {
    "allowed": {"allowed", "eligible", "approved", "qualifies", "yes", "supported", "available"},
    "not_allowed": {"not_allowed", "not allowed", "ineligible", "denied", "does_not_qualify", "does not qualify", "no", "blocked", "unavailable"},
    "insufficient_evidence": {"insufficient_evidence", "insufficient evidence", "cannot_determine", "cannot determine", "cannot_verify", "cannot verify", "unknown", "uncertain"},
    "numeric_value": {"numeric_value", "numeric", "number", "value"},
}

def normalize_decision(x: Any) -> str:
    if x is None:
        return "unknown"
    s = str(x).strip().lower().replace("-", "_")
    for canon, aliases in DECISION_ALIASES.items():
        if s == canon or s in aliases:
            return canon
    return s


def extract_json_object(text: str) -> Optional[dict]:
    if not text:
        return None
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            return None
    return None


def parse_final_answer(text: str) -> dict:
    obj = extract_json_object(text) or {}
    decision = normalize_decision(obj.get("decision")) if obj else normalize_decision(text)
    numeric_answer = obj.get("numeric_answer") if obj else None
    if numeric_answer is None:
        nums = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", text or "")
        numeric_answer = float(nums[-1]) if nums else None
    else:
        try:
            numeric_answer = float(numeric_answer)
        except Exception:
            numeric_answer = None
    return {
        "decision": decision,
        "numeric_answer": numeric_answer,
        "tool_path": obj.get("tool_path", []) if isinstance(obj, dict) else [],
        "answer": obj.get("answer", text) if isinstance(obj, dict) else text,
        "evidence_used": obj.get("evidence_used", []) if isinstance(obj, dict) else [],
        "uncertainty": obj.get("uncertainty", "unknown") if isinstance(obj, dict) else "unknown",
        "raw": obj,
    }


In [ ]:
# ============================================================
# Controlled fault injection and detection
# ============================================================

def stable_unit_interval(*parts: Any) -> float:
    """Map arbitrary input parts to a deterministic float in the interval [0, 1).

    This helper is used instead of process-level randomness so that fault
    scheduling is reproducible across notebook runs, methods, and environments.

    Args:
        *parts: Values that identify the sampling context, such as task ID,
            seed, target tool, or sampling purpose.

    Returns:
        A deterministic pseudo-random float in the interval [0, 1).
    """
    s = "|".join(str(p) for p in parts)
    h = hashlib.sha256(s.encode("utf-8")).hexdigest()[:12]
    return int(h, 16) / float(16 ** 12)


def stable_choice(options: List[str], *parts: Any) -> str:
    """Select an option deterministically from a list.

    Args:
        options: Candidate string values.
        *parts: Values used to produce the deterministic selection key.

    Returns:
        One selected option. If the options list is empty, returns "none".
    """
    if not options:
        return "none"
    u = stable_unit_interval(*parts)
    return options[int(u * len(options)) % len(options)]


def forced_fault_task_ids(intensity: float) -> set:
    """Select a deterministic subset of tasks that should receive faults.

    The selected task subset is shared across orchestration methods. This
    preserves paired comparisons: each method is exposed to the same task-level
    fault opportunities under a fixed seed and fault intensity.

    Args:
        intensity: Fault intensity used to determine the size of the forced
            fault subset.

    Returns:
        A set of task IDs selected for forced fault injection. Returns an empty
        set when intensity is zero or when no live tasks are configured.
    """
    if intensity <= 0 or not LIVE_TASKS:
        return set()

    k = max(1, int(math.ceil(intensity * len(LIVE_TASKS))))
    ranked = sorted(
        LIVE_TASKS,
        key=lambda t: stable_unit_interval(t["task_id"], "fault_rank"),
    )
    return {t["task_id"] for t in ranked[:k]}


FORCED_FAULT_TASK_IDS = forced_fault_task_ids(FAULT_INTENSITY)
print(
    "Forced fault task ids for model-in-the-loop validation:",
    sorted(FORCED_FAULT_TASK_IDS),
)


def choose_fault_type(task: dict, tool_name: str, seed: int) -> str:
    """Choose a deterministic fault type for a task, tool, and seed.

    The validation uses a compact set of production-relevant fault types:
    timeout, unavailable tool, malformed output, stale context, and
    wrong-but-plausible output.

    Args:
        task: Task specification dictionary.
        tool_name: Name of the tool selected for fault injection.
        seed: Experimental seed.

    Returns:
        A deterministic fault type from FAULT_TYPES.
    """
    return stable_choice(FAULT_TYPES, task["task_id"], tool_name, seed, "fault_type")


def make_fault_plan(task: dict, seed: int, intensity: float) -> dict:
    """Create the fault plan for a single task execution.

    The plan determines whether a fault should be injected, which required tool
    should be targeted, and which fault type should be applied. Fault decisions
    are deterministic so that methods can be compared under matched conditions.

    Args:
        task: Task specification dictionary.
        seed: Experimental seed.
        intensity: Fault intensity threshold.

    Returns:
        A dictionary with fields:
            inject: Whether a fault should be injected.
            target_tool: Tool selected for injection, or None.
            fault_type: Fault type selected for injection, or None.
    """
    required = list(task.get("required_tools", []))
    should_fault = (
        task["task_id"] in FORCED_FAULT_TASK_IDS
        or stable_unit_interval(task["task_id"], seed, "should_fault") < intensity
    )

    if not should_fault or not required:
        return {"inject": False, "target_tool": None, "fault_type": None}

    target_tool = stable_choice(required, task["task_id"], seed, "target_tool")
    fault_type = choose_fault_type(task, target_tool, seed)
    return {"inject": True, "target_tool": target_tool, "fault_type": fault_type}


def apply_fault(clean_result: dict, fault_type: Optional[str], tool_name: str) -> dict:
    """Apply one controlled fault to a clean local tool result.

    Faults are injected after the deterministic local tool returns a clean
    response. This keeps tool semantics reproducible while allowing the
    orchestrator to be evaluated under controlled runtime, schema, context, and
    semantic-output failures.

    Args:
        clean_result: Clean local tool response.
        fault_type: Fault type to inject. If None, the clean result is returned.
        tool_name: Name of the tool whose output is being faulted.

    Returns:
        A tool response dictionary containing either the original clean result
        or a controlled faulted variant.
    """
    if not fault_type:
        return clean_result

    r = deepcopy(clean_result)

    if fault_type == "timeout":
        return {
            "status": "error",
            "error_type": "timeout",
            "message": f"Injected timeout in {tool_name}",
            "evidence_source": tool_name,
            "fault_injected": "timeout",
        }

    if fault_type == "unavailable_tool":
        return {
            "status": "error",
            "error_type": "unavailable_tool",
            "message": f"Injected unavailable tool: {tool_name}",
            "evidence_source": tool_name,
            "fault_injected": "unavailable_tool",
        }

    if fault_type == "malformed_output":
        return {
            "status": "ok",
            "data": {
                "malformed": True,
                "note": "Injected malformed output with required fields missing",
            },
            "evidence_source": tool_name,
            "freshness": "current",
            "fault_injected": "malformed_output",
        }

    if fault_type == "stale_context":
        r["freshness"] = "outdated"
        r["fault_injected"] = "stale_context"

        if isinstance(r.get("data"), dict):
            r["data"] = deepcopy(r["data"])
            r["data"]["version"] = "2025-01-outdated"
            r["data"]["note"] = (
                "Injected stale context; do not treat as current evidence "
                "without refresh."
            )

            if tool_name == "policy_lookup" and "text" in r["data"]:
                r["data"]["text"] = (
                    "Outdated policy text. Current eligibility rules may have "
                    "changed; refresh before relying on this evidence."
                )

        return r

    if fault_type == "wrong_but_plausible":
        r["fault_injected"] = "wrong_but_plausible"

        if isinstance(r.get("data"), dict):
            d = deepcopy(r["data"])

            if tool_name == "calculator" and "value" in d:
                d["value"] = round(float(d["value"]) * 1.10 + 1.0, 2)
                d["note"] = "Injected plausible but wrong numeric value"

            elif tool_name == "order_lookup":
                if "delay_hours" in d:
                    d["delay_hours"] = (
                        0 if float(d.get("delay_hours") or 0) > 48 else 72
                    )
                    d["status"] = "on_time" if d["delay_hours"] == 0 else "delayed"
                d["note"] = "Injected plausible but wrong order state"

            elif tool_name == "customer_lookup":
                if d.get("tier") in {"Gold", "Platinum"}:
                    d["tier"] = "Silver"
                else:
                    d["tier"] = "Gold"
                d["note"] = "Injected plausible but wrong customer tier"

            elif tool_name == "inventory_lookup":
                d["in_stock"] = not bool(d.get("in_stock", False))
                d["available_units"] = (
                    0
                    if d.get("in_stock") is False
                    else max(1, int(d.get("available_units", 0)))
                )
                d["note"] = "Injected plausible but wrong inventory state"

            elif tool_name == "shipping_estimator":
                if "priority_allowed" in d:
                    d["priority_allowed"] = not bool(d["priority_allowed"])
                    d["reason"] = "Injected plausible but wrong shipping eligibility"
                if "estimated_days" in d:
                    d["estimated_days"] = int(d.get("estimated_days") or 3) + 4

            elif tool_name == "policy_lookup":
                if "text" in d:
                    d["text"] = (
                        "Plausible but incorrect policy text injected for "
                        "validation; verify against current operational evidence."
                    )
                d["note"] = "Injected plausible but wrong policy content"

            r["data"] = d

        return r

    return r


def detect_failure(result: dict) -> dict:
    """Detect and classify failure signals in a tool result.

    Args:
        result: Tool response dictionary, either clean or faulted.

    Returns:
        A dictionary with:
            has_failure: Whether a failure signal is present.
            signal: Specific detected signal, such as timeout or stale_context.
            failure_class: Coarser failure class used by the recovery policy.
    """
    fault = result.get("fault_injected")

    if result.get("status") == "error":
        return {
            "has_failure": True,
            "signal": result.get("error_type", "tool_error"),
            "failure_class": "tool_invocation_failure",
        }

    if fault == "malformed_output":
        return {
            "has_failure": True,
            "signal": "malformed_output",
            "failure_class": "schema_or_output_failure",
        }

    if fault == "stale_context" or result.get("freshness") == "outdated":
        return {
            "has_failure": True,
            "signal": "stale_context",
            "failure_class": "context_failure",
        }

    if fault == "wrong_but_plausible":
        return {
            "has_failure": True,
            "signal": "wrong_but_plausible",
            "failure_class": "semantic_output_failure",
        }

    return {"has_failure": False, "signal": None, "failure_class": None}


def is_explicit_runtime_failure(result: dict) -> bool:
    """Return whether a tool result represents an explicit runtime failure.

    Explicit runtime failures are different from stale, malformed, or
    semantically incorrect outputs because they are surfaced as tool errors.

    Args:
        result: Tool response dictionary.

    Returns:
        True if the response is an explicit runtime error, otherwise False.
    """
    return (
        result.get("status") == "error"
        and result.get("error_type")
        in {"timeout", "unavailable_tool", "tool_exception", "not_found"}
    )

In [ ]:
# ============================================================
# OpenAI / mock model wrapper
# ============================================================

try:
    from openai import OpenAI
except Exception:
    OpenAI = None

client = None
if not USE_MOCK_MODEL:
    if OpenAI is None:
        raise RuntimeError("openai package is not installed. Run `%pip install openai`.")
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY is missing. Set it before live execution.")
    client = OpenAI()
    print("OpenAI client initialized for live mode")
else:
    print("Running in mock-model mode. Set FORCE_MOCK_MODEL=false and OPENAI_API_KEY for live mode.")


def estimate_cost(input_tokens: int, output_tokens: int) -> float:
    return (input_tokens / 1_000_000) * INPUT_COST_PER_MILLION + (output_tokens / 1_000_000) * OUTPUT_COST_PER_MILLION


def mock_agent_response(task: dict, tool_outputs: List[dict], method: str) -> dict:
    decision = task.get("expected_decision", "unknown")
    numeric_answer = task.get("expected_numeric_answer")
    unresolved_faults = [x for x in tool_outputs if x.get("unresolved_fault")]
    if unresolved_faults and method != "self_healing":
        decision = "insufficient_evidence"
        numeric_answer = None
        answer = "Mock answer: insufficient evidence due to unresolved injected tool fault."
    elif task["success_mode"] == "numeric":
        answer = f"The numeric answer is {numeric_answer}."
    elif decision == "allowed":
        answer = "The requested action is allowed based on the required tool evidence."
    elif decision == "not_allowed":
        answer = "The requested action is not allowed based on the required tool evidence."
    else:
        answer = "The result cannot be determined from the available evidence."
    return {
        "decision": decision,
        "numeric_answer": numeric_answer,
        "tool_path": [x.get("tool_name", "") for x in tool_outputs],
        "answer": answer,
        "evidence_used": task.get("expected_facts", []),
        "uncertainty": "medium" if unresolved_faults else "none",
    }


def make_system_prompt(task: dict, method: str) -> str:
    method_note = {
        "retry_only": "If a tool returns an explicit runtime error, you may retry. Do not invent evidence.",
        "full_replanning": "If a tool result indicates failure, use the updated recovery context to form a fresh plan.",
        "self_healing": "Use recovery context when provided. Prefer verified recovered evidence over faulty evidence. Do not use stale, malformed, or contradictory evidence as current truth.",
    }.get(method, "")
    return f'''
You are a tool-using agent in a live faulted pilot benchmark.
Solve the user task using the available tools. Do not invent tool evidence.

Faults may be injected into tool outputs. If you see recovered evidence, prefer the recovered current evidence over the faulty output.

You must return your final answer as one JSON object with exactly these keys:
- decision: one of allowed, not_allowed, insufficient_evidence, numeric_value
- numeric_answer: number or null
- tool_path: list of tool names you used
- answer: concise natural-language answer
- evidence_used: list of evidence snippets from tool outputs
- uncertainty: none, low, medium, or high

For numeric tasks, set decision to numeric_value and put the numeric result in numeric_answer.
For decision tasks, set decision to allowed or not_allowed.

Orchestration method for this run: {method}.
{method_note}
'''.strip()


def live_chat_completion(messages: list, tools: Optional[list] = None, tool_choice: str = "auto", response_format: Optional[dict] = None) -> Any:
    assert client is not None
    kwargs = {"model": OPENAI_MODEL, "messages": messages}
    if tools is not None:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = tool_choice
    if response_format is not None:
        kwargs["response_format"] = response_format
    try:
        return client.chat.completions.create(**kwargs, temperature=0)
    except Exception as exc:
        if "temperature" in str(exc).lower():
            return client.chat.completions.create(**kwargs)
        raise

In [ ]:
# ============================================================
# Tool execution, recovery, and tracing
# ============================================================

@dataclass
class RunState:
    task: dict
    method: str
    seed: int
    fault_plan: dict
    recovery_budget: int = RECOVERY_BUDGET
    messages: List[dict] = field(default_factory=list)
    tool_outputs: List[dict] = field(default_factory=list)
    events: List[dict] = field(default_factory=list)
    model_calls: int = 0
    tool_calls: int = 0
    judge_calls: int = 0
    verifier_calls: int = 0
    recovery_steps: int = 0
    detected_failure: bool = False
    fault_injected: bool = False
    input_tokens: int = 0
    output_tokens: int = 0
    final_answer_text: str = ""

    def log(self, event_type: str, details: dict):
        self.events.append({"step": len(self.events) + 1, "event_type": event_type, "details": details})


def execute_clean_tool(tool_name: str, arguments: dict) -> dict:
    fn = TOOL_IMPLS.get(tool_name)
    if not fn:
        return {"status": "error", "error_type": "unknown_tool", "message": f"Unknown tool {tool_name}", "fault_injected": None}
    try:
        return fn(**arguments)
    except Exception as exc:
        return {"status": "error", "error_type": "tool_exception", "message": str(exc), "fault_injected": None}


def should_fault_this_call(state: RunState, tool_name: str) -> bool:
    return bool(state.fault_plan.get("inject") and not state.fault_injected and tool_name == state.fault_plan.get("target_tool"))


def recover_tool_result(state: RunState, tool_name: str, arguments: dict, faulted_result: dict, detection: dict) -> Tuple[dict, Optional[str]]:
    if state.recovery_budget <= 0:
        return faulted_result, None
    method = state.method
    failure_class = detection.get("failure_class")
    signal = detection.get("signal")

    if method == "retry_only":
        if is_explicit_runtime_failure(faulted_result):
            clean = execute_clean_tool(tool_name, arguments)
            state.recovery_budget -= 1
            state.recovery_steps += 1
            return {"status": "ok", "recovered": True, "recovery_action": "retry_same_tool", "faulted_result": faulted_result, "recovered_result": clean, "evidence_source": tool_name, "fault_injected": faulted_result.get("fault_injected")}, "retry_same_tool"
        return faulted_result, None

    if method == "full_replanning":
        if detection.get("has_failure"):
            clean = execute_clean_tool(tool_name, arguments)
            state.recovery_budget -= 1
            state.recovery_steps += 1
            return {"status": "ok", "recovered": True, "recovery_action": "full_replanning_reexecute", "failure_class": failure_class, "faulted_result": faulted_result, "recovered_result": clean, "evidence_source": tool_name, "fault_injected": faulted_result.get("fault_injected")}, "full_replanning_reexecute"
        return faulted_result, None

    if method == "self_healing":
        if detection.get("has_failure"):
            action_by_class = {
                "tool_invocation_failure": "targeted_retry_or_substitution",
                "schema_or_output_failure": "repair_and_recall_tool",
                "context_failure": "refresh_current_context",
                "contradiction_failure": "cross_check_and_refresh_evidence",
                "semantic_output_failure": "verify_and_recompute",
            }
            action = action_by_class.get(failure_class, "targeted_recovery")
            clean = execute_clean_tool(tool_name, arguments)
            state.recovery_budget -= 1
            state.recovery_steps += 1
            return {"status": "ok", "recovered": True, "recovery_action": action, "failure_class": failure_class, "signal": signal, "faulted_result": faulted_result, "recovered_result": clean, "evidence_source": tool_name, "fault_injected": faulted_result.get("fault_injected")}, action
    return faulted_result, None


def execute_tool_with_faults(state: RunState, tool_name: str, arguments: dict) -> dict:
    clean = execute_clean_tool(tool_name, arguments)
    faulted = clean
    if should_fault_this_call(state, tool_name):
        faulted = apply_fault(clean, state.fault_plan.get("fault_type"), tool_name)
        state.fault_injected = True
        state.log("fault_injected", {"tool_name": tool_name, "fault_type": state.fault_plan.get("fault_type"), "clean_summary": str(clean)[:250], "faulted_summary": str(faulted)[:250]})
    if TOOL_OUTPUT_VERIFIER_ENABLED:
        # Verification inspects each tool response for explicit runtime,
        # schema/output, freshness, and semantic-output failure signals.
        detection = detect_failure(faulted)
        state.verifier_calls += 1
    else:
        # Without the tool-output verifier, the orchestrator only reacts to
        # explicit runtime failures surfaced by the tool interface. It does not
        # detect stale, malformed, or plausibly wrong successful responses.
        detection = (
            detect_failure(faulted)
            if is_explicit_runtime_failure(faulted)
            else {"has_failure": False, "signal": None, "failure_class": None}
        )

    if detection.get("has_failure"):
        state.detected_failure = True
        state.log("failure_detection", {"tool_name": tool_name, **detection})
    result_to_model, action = recover_tool_result(state, tool_name, arguments, faulted, detection)
    if action:
        state.log("recovery_action", {"tool_name": tool_name, "action": action, "budget_remaining": state.recovery_budget, "result_summary": str(result_to_model)[:350]})
    unresolved_fault = bool(detection.get("has_failure") and not action)
    state.tool_outputs.append({
        "tool_name": tool_name,
        "arguments": arguments,
        "result": result_to_model,
        "clean_result": clean,
        "faulted_result": faulted if faulted != clean else None,
        "detected_failure": bool(detection.get("has_failure")),
        "recovered": bool(action),
        "unresolved_fault": unresolved_fault,
        "failure_signal": detection.get("signal"),
        "failure_class": detection.get("failure_class"),
    })
    return result_to_model


def used_tool_names(state: RunState) -> List[str]:
    return [x.get("tool_name", "") for x in state.tool_outputs]

In [ ]:
# ============================================================
# Agent execution loop
# ============================================================


def run_agent_once(task: dict, method: str, seed: int) -> Tuple[dict, dict]:
    random.seed(seed)
    fault_plan = make_fault_plan(task, seed, FAULT_INTENSITY)
    state = RunState(task=task, method=method, seed=seed, fault_plan=fault_plan)
    run_id = f"{task['task_id']}__{method}__lambda_{FAULT_INTENSITY}__seed_{seed}__budget_{RECOVERY_BUDGET}"
    state.log("run_start", {"run_id": run_id, "task_id": task["task_id"], "method": method, "fault_intensity": FAULT_INTENSITY, "seed": seed, "fault_plan": fault_plan})

    if USE_MOCK_MODEL:
        for tname in task.get("required_tools", []):
            args = infer_tool_args(tname, task)
            result = execute_tool_with_faults(state, tname, args)
            state.tool_calls += 1
            state.log("tool_call", {"tool_name": tname, "arguments": args, "result_summary": str(result)[:400]})
        final_obj = mock_agent_response(task, state.tool_outputs, method)
        state.final_answer_text = json.dumps(final_obj)
        state.model_calls = 1
        state.log("final_answer", {"answer": state.final_answer_text})
    else:
        messages = [
            {"role": "system", "content": make_system_prompt(task, method)},
            {"role": "user", "content": task["user_request"]},
        ]
        for step in range(MAX_AGENT_STEPS):
            start = time.time()
            resp = live_chat_completion(messages, tools=OPENAI_TOOLS, tool_choice="auto")
            elapsed = time.time() - start
            msg = resp.choices[0].message
            state.model_calls += 1
            usage = getattr(resp, "usage", None)
            if usage:
                state.input_tokens += int(getattr(usage, "prompt_tokens", 0) or 0)
                state.output_tokens += int(getattr(usage, "completion_tokens", 0) or 0)
            tool_calls = getattr(msg, "tool_calls", None) or []
            state.log("model_call", {"elapsed": elapsed, "has_tool_calls": bool(tool_calls), "content_preview": str(getattr(msg, "content", ""))[:250]})
            messages.append(msg)
            if tool_calls:
                for tc in tool_calls:
                    tname = tc.function.name
                    try:
                        args = json.loads(tc.function.arguments or "{}")
                    except Exception:
                        args = {}
                    result = execute_tool_with_faults(state, tname, args)
                    state.tool_calls += 1
                    state.log("tool_call", {"tool_name": tname, "arguments": args, "result_summary": str(result)[:500]})
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
                continue
            content = msg.content or ""
            state.final_answer_text = content
            state.log("final_answer", {"answer": content})
            break
        if not state.final_answer_text:
            state.final_answer_text = json.dumps({"decision": "insufficient_evidence", "numeric_answer": None, "tool_path": used_tool_names(state), "answer": "No final answer produced before max steps.", "evidence_used": [], "uncertainty": "high"})
            state.log("final_answer", {"answer": state.final_answer_text, "forced": True})

    final_parsed = parse_final_answer(state.final_answer_text)

    # Numeric answers are only meaningful for numeric tasks. For decision tasks,
    # prevent fallback parsing from extracting incidental numbers such as policy
    # thresholds, delay hours, order IDs, or dates from the final answer text.
    if task.get("success_mode") != "numeric":
        final_parsed["numeric_answer"] = None
        if isinstance(final_parsed.get("raw"), dict):
            final_parsed["raw"]["numeric_answer"] = None

    rule_result = rule_evaluate(task, state, final_parsed)
    judge = judge_answer(task, state, final_parsed, rule_result)
    row = finalize_row(run_id, task, method, seed, state, final_parsed, rule_result, judge)
    trace = {"run_id": run_id, "task": task, "row": row, "events": state.events, "tool_outputs": state.tool_outputs, "final_parsed": final_parsed, "rule_result": rule_result, "judge": judge}
    return row, trace


def infer_tool_args(tool_name: str, task: dict) -> dict:
    text = task.get("user_request", "")
    if tool_name == "order_lookup":
        oid = re.search(r"O-\d+", text)
        return {"order_id": oid.group(0) if oid else "O-1001"}
    if tool_name == "customer_lookup":
        oid = re.search(r"O-\d+", text)
        order = ORDERS.get(oid.group(0), ORDERS["O-1001"]) if oid else ORDERS["O-1001"]
        return {"customer_id": order["customer_id"]}
    if tool_name == "inventory_lookup":
        oid = re.search(r"O-\d+", text)
        order = ORDERS.get(oid.group(0), ORDERS["O-2001"]) if oid else ORDERS["O-2001"]
        return {"sku": order["sku"]}
    if tool_name == "shipping_estimator":
        oid = re.search(r"O-\d+", text)
        order = ORDERS.get(oid.group(0), ORDERS["O-2001"]) if oid else ORDERS["O-2001"]
        return {"sku": order["sku"], "destination": order.get("destination", "standard_zone"), "requested_shipping": order.get("requested_shipping", "standard"), "quantity": int(order.get("quantity", 1))}
    if tool_name == "policy_lookup":
        if "shipping" in text.lower():
            return {"policy_name": "shipping_policy"}
        if "return" in text.lower():
            return {"policy_name": "return_policy"}
        return {"policy_name": "replacement_policy"}
    if tool_name == "calculator":
        return {"expression": str(task.get("expected_numeric_answer", 0))}
    return {}

In [ ]:
# ============================================================
# Evaluation: rule checks + optional LLM judge
# ============================================================


def rule_evaluate(task: dict, state: RunState, final_parsed: dict) -> dict:
    mode = task["success_mode"]
    predicted_decision = normalize_decision(final_parsed.get("decision"))
    expected_decision = normalize_decision(task.get("expected_decision"))
    decision_match = predicted_decision == expected_decision

    used = set(used_tool_names(state))
    required = set(task.get("required_tools", []))
    missing = sorted(required - used)
    extra = sorted(used - required)
    tools_ok = len(missing) == 0
    tool_efficiency = (len(required) / len(used)) if used else 0.0

    numeric_ok = True
    if mode == "numeric":
        expected_num = task.get("expected_numeric_answer")
        tol = task.get("numeric_tolerance", 0.01)
        pred_num = final_parsed.get("numeric_answer")
        numeric_ok = pred_num is not None and expected_num is not None and abs(float(pred_num) - float(expected_num)) <= tol
        decision_match = predicted_decision == "numeric_value"

    safe_degradation = predicted_decision == "insufficient_evidence"
    rule_success = bool(decision_match and numeric_ok and tools_ok and not safe_degradation)
    return {
        "rule_success": rule_success,
        "expected_decision": expected_decision,
        "predicted_decision": predicted_decision,
        "decision_match": bool(decision_match),
        "numeric_ok": bool(numeric_ok),
        "tools_ok": bool(tools_ok),
        "missing_required_tools": "|".join(missing),
        "extra_tools_used": "|".join(extra),
        "tool_efficiency": tool_efficiency,
        "safe_degradation": bool(safe_degradation),
        "rule_reason": f"mode={mode}; decision_match={decision_match}; numeric_ok={numeric_ok}; tools_ok={tools_ok}; missing={missing}; extra={extra}",
    }


def mock_judge(task: dict, state: RunState, final_parsed: dict, rule_result: dict) -> dict:
    unresolved = any(x.get("unresolved_fault") for x in state.tool_outputs)
    if rule_result["rule_success"] and not unresolved:
        return {"outcome": "correct", "decision_match": True, "grounded_in_tool_evidence": True, "uses_stale_or_contradictory_evidence": False, "is_safe_degradation": False, "is_silent_failure": False, "confidence": 0.99, "reason": "Mock judge: rule checks passed and no unresolved fault."}
    if rule_result["safe_degradation"]:
        return {"outcome": "safe_degradation", "decision_match": False, "grounded_in_tool_evidence": True, "uses_stale_or_contradictory_evidence": unresolved, "is_safe_degradation": True, "is_silent_failure": False, "confidence": 0.8, "reason": "Mock judge: safe degradation due to unresolved fault."}
    silent_failure = (
        bool(rule_result["decision_match"])
        and not bool(rule_result["safe_degradation"])
        and bool(unresolved)
    )

    return {
        "outcome": "incorrect",
        "decision_match": bool(rule_result["decision_match"]),
        "grounded_in_tool_evidence": bool(rule_result["tools_ok"] and not unresolved),
        "uses_stale_or_contradictory_evidence": bool(unresolved),
        "is_safe_degradation": False,
        "is_silent_failure": bool(silent_failure),
        "confidence": 0.8,
        "reason": "Mock judge: rule checks failed or unresolved fault remained.",
    }


def judge_answer(task: dict, state: RunState, final_parsed: dict, rule_result: dict) -> dict:
    if USE_MOCK_MODEL or not USE_LLM_EVALUATOR:
        return mock_judge(task, state, final_parsed, rule_result)

    evidence = []
    for item in state.tool_outputs:
        evidence.append({"tool_name": item["tool_name"], "arguments": item.get("arguments"), "result_sent_to_model": item.get("result"), "detected_failure": item.get("detected_failure"), "recovered": item.get("recovered"), "unresolved_fault": item.get("unresolved_fault")})

    judge_prompt = {
        "task": {"task_id": task["task_id"], "category": task["category"], "success_mode": task["success_mode"], "user_request": task["user_request"], "expected_decision": task.get("expected_decision"), "expected_numeric_answer": task.get("expected_numeric_answer"), "expected_facts": task.get("expected_facts", []), "required_tools": task.get("required_tools", [])},
        "final_answer": final_parsed,
        "tool_evidence": evidence,
        "fault_plan": state.fault_plan,
        "detected_failure": state.detected_failure,
        "recovery_steps": state.recovery_steps,
        "rule_result": rule_result,
    }
    messages = [
        {"role": "system", "content": "You are a strict but fair evaluator for a model-in-the-loop tool-use fault-injection benchmark. Return JSON only."},
        {"role": "user", "content": "Evaluate whether the final answer is correct, decision-aligned, and grounded in current/recovered tool evidence. Faulty, stale, malformed, contradictory, or unrecovered outputs should not be treated as valid current evidence. Return JSON with keys: outcome, decision_match, grounded_in_tool_evidence, uses_stale_or_contradictory_evidence, is_safe_degradation, is_silent_failure, confidence, reason. outcome must be correct, incorrect, safe_degradation, or ambiguous.\n\n" + json.dumps(judge_prompt, indent=2)},
    ]
    start = time.time()
    try:
        resp = client.chat.completions.create(model=OPENAI_VERIFIER_MODEL, messages=messages, response_format={"type": "json_object"})
    except Exception:
        resp = client.chat.completions.create(model=OPENAI_VERIFIER_MODEL, messages=messages)
    elapsed = time.time() - start
    state.judge_calls += 1
    usage = getattr(resp, "usage", None)
    if usage:
        state.input_tokens += int(getattr(usage, "prompt_tokens", 0) or 0)
        state.output_tokens += int(getattr(usage, "completion_tokens", 0) or 0)
    content = resp.choices[0].message.content or "{}"
    judge = extract_json_object(content) or mock_judge(task, state, final_parsed, rule_result)
    judge.setdefault("outcome", "ambiguous")
    judge.setdefault("decision_match", rule_result["decision_match"])
    judge.setdefault("grounded_in_tool_evidence", rule_result["tools_ok"])
    judge.setdefault("uses_stale_or_contradictory_evidence", False)
    judge.setdefault("is_safe_degradation", rule_result["safe_degradation"])
    judge.setdefault("is_silent_failure", False)
    judge.setdefault("confidence", 0.5)
    judge.setdefault("reason", "No reason provided")
    state.log("llm_judge", {"elapsed": elapsed, "judge": judge})
    return judge

In [ ]:
# ============================================================
# Row finalization and summaries
# ============================================================


def finalize_row(run_id: str, task: dict, method: str, seed: int, state: RunState, final_parsed: dict, rule_result: dict, judge: dict) -> dict:
    judge_outcome = str(judge.get("outcome", "ambiguous"))
    task_success = bool(rule_result["rule_success"] and judge_outcome in {"correct", "ambiguous"})
    safe_degradation = bool(judge.get("is_safe_degradation", False) or rule_result.get("safe_degradation", False) or judge_outcome == "safe_degradation")
    silent_failure = bool((not task_success) and (not safe_degradation) and (not state.detected_failure))
    recovery_success = bool(task_success and state.detected_failure and state.recovery_steps > 0)
    row = {
        "run_id": run_id,
        "task_id": task["task_id"],
        "category": task["category"],
        "success_mode": task["success_mode"],
        "method": method,
        "model": OPENAI_MODEL,
        "fault_intensity": FAULT_INTENSITY,
        "seed": seed,
        "recovery_budget": RECOVERY_BUDGET,
        "fault_scheduled": bool(state.fault_plan.get("inject")),
        "fault_target_tool": state.fault_plan.get("target_tool"),
        "fault_type": state.fault_plan.get("fault_type"),
        "fault_injected": bool(state.fault_injected),
        "expected_decision": rule_result["expected_decision"],
        "predicted_decision": rule_result["predicted_decision"],
        "decision_match": rule_result["decision_match"],
        "numeric_answer": final_parsed.get("numeric_answer") if task.get("success_mode") == "numeric" else None,
        "expected_numeric_answer": task.get("expected_numeric_answer"),
        "numeric_ok": rule_result["numeric_ok"],
        "tools_ok": rule_result["tools_ok"],
        "missing_required_tools": rule_result["missing_required_tools"],
        "extra_tools_used": rule_result["extra_tools_used"],
        "tool_efficiency": rule_result["tool_efficiency"],
        "task_success": task_success,
        "silent_failure": silent_failure,
        "safe_degradation": safe_degradation,
        "detected_failure": bool(state.detected_failure),
        "recovery_success": recovery_success,
        "llm_judge_outcome": judge_outcome,
        "judge_confidence": float(judge.get("confidence", 0.0) or 0.0),
        "grounded_in_tool_evidence": bool(judge.get("grounded_in_tool_evidence", False)),
        "uses_stale_or_contradictory_evidence": bool(judge.get("uses_stale_or_contradictory_evidence", False)),
        "model_calls": state.model_calls,
        "tool_calls": state.tool_calls,
        "verifier_calls": state.verifier_calls,
        "judge_calls": state.judge_calls,
        "recovery_steps": state.recovery_steps,
        "input_tokens": state.input_tokens,
        "output_tokens": state.output_tokens,
        "estimated_cost_usd": estimate_cost(state.input_tokens, state.output_tokens),
        "eval_reason": f"rule=({rule_result['rule_reason']}); judge={judge_outcome}; {judge.get('reason', '')}",
        "final_answer": final_parsed.get("answer", ""),
        "tool_path": "|".join([str(x).replace("functions.", "") for x in final_parsed.get("tool_path", [])]),
        "tools_used": "|".join(used_tool_names(state)),
    }
    return row


def summarize_by(df: pd.DataFrame, group_cols: List[str]) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    g = df.groupby(group_cols, dropna=False)
    out = g.agg(
        executions=("run_id", "count"),
        success_rate=("task_success", "mean"),
        silent_failure_rate=("silent_failure", "mean"),
        safe_degradation_rate=("safe_degradation", "mean"),
        detected_failure_rate=("detected_failure", "mean"),
        recovery_success_rate=("recovery_success", "mean"),
        fault_scheduled_rate=("fault_scheduled", "mean"),
        fault_injected_rate=("fault_injected", "mean"),
        avg_model_calls=("model_calls", "mean"),
        avg_tool_calls=("tool_calls", "mean"),
        avg_judge_calls=("judge_calls", "mean"),
        avg_recovery_steps=("recovery_steps", "mean"),
        avg_input_tokens=("input_tokens", "mean"),
        avg_output_tokens=("output_tokens", "mean"),
        avg_cost_usd=("estimated_cost_usd", "mean"),
    ).reset_index()
    return out

## Run supplemental experiments

The supplemental experiments reuse the same task set, tools, fault injector, recovery policies, and evaluation pipeline as the main validation.

### Verifier ablation

The verifier ablation compares two variants of the self-healing orchestrator:

| Variant | Tool-output verifier | Interpretation |
|---|---:|---|
| `verifier_on` | enabled | Detects explicit runtime, schema/output, context, and semantic-output faults |
| `verifier_off` | disabled | Reacts only to explicit runtime failures; stale/malformed/plausibly wrong successful outputs are not detected |

### Recovery-budget sensitivity

The budget-sensitivity experiment varies the recovery budget while keeping the task set, seeds, fault intensity, and methods fixed.


In [ ]:
# ============================================================
# Supplemental run helpers
# ============================================================

def run_with_settings(
    task: dict,
    method: str,
    seed: int,
    *,
    experiment: str,
    variant: str,
    recovery_budget: int,
    verifier_enabled: bool,
) -> Tuple[dict, dict]:
    """Run one task under a temporary verifier and recovery-budget setting.

    Args:
        task: Task specification dictionary.
        method: Orchestration method.
        seed: Experimental seed.
        experiment: Experiment label stored in result rows.
        variant: Variant label stored in result rows.
        recovery_budget: Recovery budget for this run.
        verifier_enabled: Whether tool-output verification is enabled.

    Returns:
        A pair of result row and trace dictionary.
    """
    global RECOVERY_BUDGET, TOOL_OUTPUT_VERIFIER_ENABLED

    previous_budget = RECOVERY_BUDGET
    previous_verifier = TOOL_OUTPUT_VERIFIER_ENABLED

    try:
        RECOVERY_BUDGET = recovery_budget
        TOOL_OUTPUT_VERIFIER_ENABLED = verifier_enabled

        row, trace = run_agent_once(task, method, seed)

        original_run_id = row["run_id"]
        run_id = f"{experiment}__{variant}__{original_run_id}"

        row["run_id"] = run_id
        row["experiment"] = experiment
        row["variant"] = variant
        row["tool_output_verifier_enabled"] = bool(verifier_enabled)
        row["recovery_budget_level"] = recovery_budget

        trace["run_id"] = run_id
        trace["experiment"] = experiment
        trace["variant"] = variant
        trace["tool_output_verifier_enabled"] = bool(verifier_enabled)
        trace["recovery_budget_level"] = recovery_budget
        trace["row"] = row

        return row, trace
    finally:
        RECOVERY_BUDGET = previous_budget
        TOOL_OUTPUT_VERIFIER_ENABLED = previous_verifier


In [ ]:
# ============================================================
# Verifier ablation
# ============================================================

ablation_rows = []
ablation_traces = []

if RUN_VERIFIER_ABLATION:
    ablation_variants = [
        {"variant": "verifier_on", "verifier_enabled": True},
        {"variant": "verifier_off", "verifier_enabled": False},
    ]

    total = len(LIVE_TASKS) * len(ablation_variants) * len(SUPPLEMENTAL_SEEDS)
    print(f"Planned verifier-ablation runs: {total}")

    i = 0
    for task in LIVE_TASKS:
        for variant_cfg in ablation_variants:
            for seed in SUPPLEMENTAL_SEEDS:
                i += 1
                print(
                    f"[ablation {i}/{total}] "
                    f"{task['task_id']} | {variant_cfg['variant']} | seed={seed}"
                )
                row, trace = run_with_settings(
                    task,
                    "self_healing",
                    seed,
                    experiment="verifier_ablation",
                    variant=variant_cfg["variant"],
                    recovery_budget=RECOVERY_BUDGET,
                    verifier_enabled=variant_cfg["verifier_enabled"],
                )
                ablation_rows.append(row)
                ablation_traces.append(trace)
else:
    print("RUN_VERIFIER_ABLATION=False; skipping verifier ablation.")

ablation_df = pd.DataFrame(ablation_rows)
print("Verifier ablation rows:", len(ablation_df))
display(ablation_df.head())


In [ ]:
# ============================================================
# Recovery-budget sensitivity
# ============================================================

budget_rows = []
budget_traces = []

if RUN_BUDGET_SENSITIVITY:
    budget_tasks = LIVE_TASKS[:BUDGET_TASK_LIMIT]
    total = len(budget_tasks) * len(METHODS) * len(BUDGET_LEVELS) * len(SUPPLEMENTAL_SEEDS)
    print(f"Planned budget-sensitivity runs: {total}")

    i = 0
    for task in budget_tasks:
        for method in METHODS:
            for budget in BUDGET_LEVELS:
                for seed in SUPPLEMENTAL_SEEDS:
                    i += 1
                    print(
                        f"[budget {i}/{total}] "
                        f"{task['task_id']} | {method} | budget={budget} | seed={seed}"
                    )
                    row, trace = run_with_settings(
                        task,
                        method,
                        seed,
                        experiment="budget_sensitivity",
                        variant=f"budget_{budget}",
                        recovery_budget=budget,
                        verifier_enabled=True,
                    )
                    budget_rows.append(row)
                    budget_traces.append(trace)
else:
    print("RUN_BUDGET_SENSITIVITY=False; skipping budget sensitivity.")

budget_df = pd.DataFrame(budget_rows)
print("Budget-sensitivity rows:", len(budget_df))
display(budget_df.head())


In [ ]:
# ============================================================
# Supplemental summaries and health checks
# ============================================================

ablation_summary_by_variant = (
    summarize_by(ablation_df, ["variant"])
    if not ablation_df.empty
    else pd.DataFrame()
)
ablation_summary_by_fault_type = (
    summarize_by(ablation_df[ablation_df["fault_injected"]], ["variant", "fault_type"])
    if not ablation_df.empty and ablation_df["fault_injected"].any()
    else pd.DataFrame()
)
ablation_failed_rows = (
    ablation_df[~ablation_df["task_success"]].copy()
    if not ablation_df.empty
    else pd.DataFrame()
)
ablation_recovery_rows = (
    ablation_df[ablation_df["detected_failure"] | ablation_df["recovery_success"] | ablation_df["fault_injected"]].copy()
    if not ablation_df.empty
    else pd.DataFrame()
)

budget_summary_by_method_budget = (
    summarize_by(budget_df, ["method", "recovery_budget_level"])
    if not budget_df.empty
    else pd.DataFrame()
)
budget_summary_by_fault_type = (
    summarize_by(budget_df[budget_df["fault_injected"]], ["method", "recovery_budget_level", "fault_type"])
    if not budget_df.empty and budget_df["fault_injected"].any()
    else pd.DataFrame()
)
budget_failed_rows = (
    budget_df[~budget_df["task_success"]].copy()
    if not budget_df.empty
    else pd.DataFrame()
)
budget_recovery_rows = (
    budget_df[budget_df["detected_failure"] | budget_df["recovery_success"] | budget_df["fault_injected"]].copy()
    if not budget_df.empty
    else pd.DataFrame()
)

health_rows = []

if not ablation_df.empty:
    for _, r in ablation_summary_by_variant.iterrows():
        health_rows.append({
            "experiment": "verifier_ablation",
            "group": r.get("variant"),
            "executions": int(r["executions"]),
            "success_rate": float(r["success_rate"]),
            "silent_failure_rate": float(r["silent_failure_rate"]),
            "detected_failure_rate": float(r["detected_failure_rate"]),
            "recovery_success_rate": float(r["recovery_success_rate"]),
            "fault_injected_rate": float(r["fault_injected_rate"]),
            "diagnostic_no_faults": bool(r["fault_injected_rate"] == 0),
            "diagnostic_all_success": bool(r["success_rate"] == 1.0),
        })

if not budget_df.empty:
    for _, r in budget_summary_by_method_budget.iterrows():
        health_rows.append({
            "experiment": "budget_sensitivity",
            "group": f"{r.get('method')}__budget_{r.get('recovery_budget_level')}",
            "executions": int(r["executions"]),
            "success_rate": float(r["success_rate"]),
            "silent_failure_rate": float(r["silent_failure_rate"]),
            "detected_failure_rate": float(r["detected_failure_rate"]),
            "recovery_success_rate": float(r["recovery_success_rate"]),
            "fault_injected_rate": float(r["fault_injected_rate"]),
            "diagnostic_no_faults": bool(r["fault_injected_rate"] == 0),
            "diagnostic_all_success": bool(r["success_rate"] == 1.0),
        })

supplemental_health_check = pd.DataFrame(health_rows)

print("Verifier ablation summary")
display(ablation_summary_by_variant)

print("Verifier ablation by fault type")
display(ablation_summary_by_fault_type)

print("Budget sensitivity summary")
display(budget_summary_by_method_budget)

print("Supplemental health check")
display(supplemental_health_check)


In [ ]:
# ============================================================
# Export CSVs, traces, and tables
# ============================================================

prefix = "model_in_loop_supplemental_experiments"

csv_paths = {}
table_paths = {}
trace_paths = {}

if not ablation_df.empty:
    csv_paths["verifier_ablation_runs"] = RESULTS_DIR / f"{prefix}_verifier_ablation_runs.csv"
    csv_paths["verifier_ablation_summary_by_variant"] = RESULTS_DIR / f"{prefix}_verifier_ablation_summary_by_variant.csv"
    csv_paths["verifier_ablation_summary_by_fault_type"] = RESULTS_DIR / f"{prefix}_verifier_ablation_summary_by_fault_type.csv"
    csv_paths["verifier_ablation_failed_rows"] = RESULTS_DIR / f"{prefix}_verifier_ablation_failed_rows.csv"
    csv_paths["verifier_ablation_recovery_rows"] = RESULTS_DIR / f"{prefix}_verifier_ablation_recovery_rows.csv"

    ablation_df.to_csv(csv_paths["verifier_ablation_runs"], index=False)
    ablation_summary_by_variant.to_csv(csv_paths["verifier_ablation_summary_by_variant"], index=False)
    ablation_summary_by_fault_type.to_csv(csv_paths["verifier_ablation_summary_by_fault_type"], index=False)
    ablation_failed_rows.to_csv(csv_paths["verifier_ablation_failed_rows"], index=False)
    ablation_recovery_rows.to_csv(csv_paths["verifier_ablation_recovery_rows"], index=False)

    table_paths["verifier_ablation_summary"] = TABLES_DIR / f"{prefix}_verifier_ablation_summary_table.tex"
    ablation_summary_by_variant.to_latex(table_paths["verifier_ablation_summary"], index=False, float_format="%.3f")

    trace_paths["verifier_ablation_all_traces"] = TRACES_DIR / f"{prefix}_verifier_ablation_all_traces.json"
    with open(trace_paths["verifier_ablation_all_traces"], "w") as f:
        json.dump(ablation_traces, f, indent=2)

    if ablation_traces:
        trace_paths["verifier_ablation_representative_trace"] = TRACES_DIR / f"{prefix}_verifier_ablation_representative_trace.json"
        with open(trace_paths["verifier_ablation_representative_trace"], "w") as f:
            json.dump(ablation_traces[0], f, indent=2)

if not budget_df.empty:
    csv_paths["budget_sensitivity_runs"] = RESULTS_DIR / f"{prefix}_budget_sensitivity_runs.csv"
    csv_paths["budget_sensitivity_summary_by_method_budget"] = RESULTS_DIR / f"{prefix}_budget_sensitivity_summary_by_method_budget.csv"
    csv_paths["budget_sensitivity_summary_by_fault_type"] = RESULTS_DIR / f"{prefix}_budget_sensitivity_summary_by_fault_type.csv"
    csv_paths["budget_sensitivity_failed_rows"] = RESULTS_DIR / f"{prefix}_budget_sensitivity_failed_rows.csv"
    csv_paths["budget_sensitivity_recovery_rows"] = RESULTS_DIR / f"{prefix}_budget_sensitivity_recovery_rows.csv"

    budget_df.to_csv(csv_paths["budget_sensitivity_runs"], index=False)
    budget_summary_by_method_budget.to_csv(csv_paths["budget_sensitivity_summary_by_method_budget"], index=False)
    budget_summary_by_fault_type.to_csv(csv_paths["budget_sensitivity_summary_by_fault_type"], index=False)
    budget_failed_rows.to_csv(csv_paths["budget_sensitivity_failed_rows"], index=False)
    budget_recovery_rows.to_csv(csv_paths["budget_sensitivity_recovery_rows"], index=False)

    table_paths["budget_sensitivity_summary"] = TABLES_DIR / f"{prefix}_budget_sensitivity_summary_table.tex"
    budget_summary_by_method_budget.to_latex(table_paths["budget_sensitivity_summary"], index=False, float_format="%.3f")

    trace_paths["budget_sensitivity_all_traces"] = TRACES_DIR / f"{prefix}_budget_sensitivity_all_traces.json"
    with open(trace_paths["budget_sensitivity_all_traces"], "w") as f:
        json.dump(budget_traces, f, indent=2)

    if budget_traces:
        trace_paths["budget_sensitivity_representative_trace"] = TRACES_DIR / f"{prefix}_budget_sensitivity_representative_trace.json"
        with open(trace_paths["budget_sensitivity_representative_trace"], "w") as f:
            json.dump(budget_traces[0], f, indent=2)

if not supplemental_health_check.empty:
    csv_paths["supplemental_health_check"] = RESULTS_DIR / f"{prefix}_health_check.csv"
    supplemental_health_check.to_csv(csv_paths["supplemental_health_check"], index=False)

    table_paths["supplemental_health_check"] = TABLES_DIR / f"{prefix}_health_check_table.tex"
    supplemental_health_check.to_latex(table_paths["supplemental_health_check"], index=False, float_format="%.3f")

print("Exports complete")
print("Results:", list(csv_paths.values()))
print("Tables:", list(table_paths.values()))
print("Traces:", list(trace_paths.values()))


In [ ]:
# ============================================================
# Figures
# ============================================================

prefix = "model_in_loop_supplemental_experiments"
figure_paths = {}

if not ablation_summary_by_variant.empty:
    plot_df = ablation_summary_by_variant.sort_values("variant")

    plt.figure(figsize=(7, 4))
    plt.bar(plot_df["variant"], plot_df["success_rate"])
    plt.ylim(0, 1)
    plt.ylabel("Success rate")
    plt.title("Verifier ablation: success by variant")
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    p = FIGURES_DIR / f"{prefix}_verifier_ablation_success_by_variant.png"
    plt.savefig(p, dpi=200)
    figure_paths["verifier_ablation_success_by_variant"] = p
    plt.show()

    x = np.arange(len(plot_df))
    width = 0.35

    plt.figure(figsize=(8, 4))
    plt.bar(x - width / 2, plot_df["recovery_success_rate"], width=width, label="recovery success")
    plt.bar(x + width / 2, plot_df["silent_failure_rate"], width=width, label="silent failure")
    plt.xticks(x, plot_df["variant"], rotation=15, ha="right")
    plt.ylim(0, 1)
    plt.ylabel("Rate")
    plt.title("Verifier ablation: recovery and silent-failure rates")
    plt.legend()
    plt.tight_layout()
    p = FIGURES_DIR / f"{prefix}_verifier_ablation_recovery_vs_silent_failure.png"
    plt.savefig(p, dpi=200)
    figure_paths["verifier_ablation_recovery_vs_silent_failure"] = p
    plt.show()

if not budget_summary_by_method_budget.empty:
    plot_df = budget_summary_by_method_budget.copy()

    plt.figure(figsize=(8, 4))
    for method in sorted(plot_df["method"].unique()):
        sub = plot_df[plot_df["method"] == method].sort_values("recovery_budget_level")
        plt.plot(sub["recovery_budget_level"], sub["success_rate"], marker="o", label=method)
    plt.xlabel("Recovery budget")
    plt.ylabel("Success rate")
    plt.ylim(0, 1)
    plt.title("Budget sensitivity: success vs recovery budget")
    plt.legend()
    plt.tight_layout()
    p = FIGURES_DIR / f"{prefix}_budget_sensitivity_success_vs_budget.png"
    plt.savefig(p, dpi=200)
    figure_paths["budget_sensitivity_success_vs_budget"] = p
    plt.show()

    plt.figure(figsize=(8, 4))
    for method in sorted(plot_df["method"].unique()):
        sub = plot_df[plot_df["method"] == method].sort_values("recovery_budget_level")
        plt.plot(sub["recovery_budget_level"], sub["recovery_success_rate"], marker="o", label=method)
    plt.xlabel("Recovery budget")
    plt.ylabel("Recovery success rate")
    plt.ylim(0, 1)
    plt.title("Budget sensitivity: recovery success vs recovery budget")
    plt.legend()
    plt.tight_layout()
    p = FIGURES_DIR / f"{prefix}_budget_sensitivity_recovery_success_vs_budget.png"
    plt.savefig(p, dpi=200)
    figure_paths["budget_sensitivity_recovery_success_vs_budget"] = p
    plt.show()

    plt.figure(figsize=(8, 4))
    for method in sorted(plot_df["method"].unique()):
        sub = plot_df[plot_df["method"] == method].sort_values("recovery_budget_level")
        call_proxy = sub["avg_model_calls"] + sub["avg_tool_calls"] + sub["avg_judge_calls"]
        plt.plot(sub["recovery_budget_level"], call_proxy, marker="o", label=method)
    plt.xlabel("Recovery budget")
    plt.ylabel("Average call-count proxy")
    plt.title("Budget sensitivity: call-count proxy vs recovery budget")
    plt.legend()
    plt.tight_layout()
    p = FIGURES_DIR / f"{prefix}_budget_sensitivity_call_proxy_vs_budget.png"
    plt.savefig(p, dpi=200)
    figure_paths["budget_sensitivity_call_proxy_vs_budget"] = p
    plt.show()

print("Figures:", list(figure_paths.values()))


In [ ]:
# ============================================================
# Artifact manifest
# ============================================================

prefix = "model_in_loop_supplemental_experiments"

manifest = {
    "notebook": "model_in_the_loop_supplemental_experiments.ipynb",
    "purpose": (
        "Supplemental model-in-the-loop validation experiments for verifier "
        "ablation and recovery-budget sensitivity."
    ),
    "model": OPENAI_MODEL,
    "verifier_model": OPENAI_VERIFIER_MODEL,
    "use_mock_model": USE_MOCK_MODEL,
    "use_llm_evaluator": USE_LLM_EVALUATOR,
    "fault_intensity": FAULT_INTENSITY,
    "fault_types": FAULT_TYPES,
    "seeds": SUPPLEMENTAL_SEEDS,
    "run_verifier_ablation": RUN_VERIFIER_ABLATION,
    "run_budget_sensitivity": RUN_BUDGET_SENSITIVITY,
    "budget_levels": BUDGET_LEVELS,
    "budget_task_limit": BUDGET_TASK_LIMIT,
    "rows": {
        "verifier_ablation": int(len(ablation_df)),
        "budget_sensitivity": int(len(budget_df)),
    },
    "outputs": {
        "results": {k: str(v) for k, v in csv_paths.items()},
        "tables": {k: str(v) for k, v in table_paths.items()},
        "figures": {k: str(v) for k, v in figure_paths.items()},
        "traces": {k: str(v) for k, v in trace_paths.items()},
    },
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}

manifest_path = BASE_DIR / f"{prefix}_artifact_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

manifest


## Interpretation guide

Use the verifier ablation to answer:

> Does tool-output verification help the self-healing orchestrator detect and recover from stale, malformed, or plausibly wrong successful tool outputs?

Use the budget-sensitivity experiment to answer:

> Does additional recovery budget improve reliability, and does it create meaningful call-count overhead?

Recommended reporting:

- Treat the verifier ablation as the stronger supplemental result because it directly tests the value of runtime verification.
- Treat budget sensitivity as a secondary diagnostic result showing whether reliability gains come from targeted recovery or simply from more attempts.
- Report success rate, recovery success rate, silent-failure rate, and average call-count proxy.
- Do not overclaim. These are compact supplemental validations, not broad multi-model benchmarks.
